# Decision Tree Classifier

A Decision Tree Classifier recursively splits the feature space using **information gain** (reduction in entropy) to separate class labels. Predictions at each leaf are the majority class.

**Dataset:** Breast Cancer Wisconsin — classify tumours as Malignant or Benign.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, '.')
from decision_tree_classifier import decision_tree_classifier
np.random.seed(42)
print("Imports complete")

## Load & Explore the Data

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
class_names   = data.target_names

print(f"Samples:  {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes:  {list(class_names)}  (0=Malignant, 1=Benign)")
print(f"Class distribution: Malignant={( y==0).sum()}, Benign={(y==1).sum()}")

## Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

## Effect of Max Depth

Deeper trees fit training data more closely but may overfit. We sweep max_depth to see the trade-off.

In [ ]:
depths = list(range(1, 12))
train_acc, test_acc = [], []

for d in depths:
    clf = decision_tree_classifier(max_depth=d)
    clf.fit(X_train, y_train)
    train_acc.append(clf.score(X_train, y_train))
    test_acc.append(clf.score(X_test, y_test))

plt.figure(figsize=(9,5))
plt.plot(depths, train_acc, 'o-', color='steelblue', label='Train accuracy')
plt.plot(depths, test_acc,  's-', color='tomato',    label='Test accuracy')
plt.xlabel("Max Depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree Depth vs Accuracy")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

best_depth = depths[test_acc.index(max(test_acc))]
print(f"Best test accuracy {max(test_acc):.4f} at max_depth={best_depth}")

## Train Best Model

In [ ]:
clf = decision_tree_classifier(max_depth=4)
clf.fit(X_train, y_train)

print(f"Train accuracy: {clf.score(X_train, y_train):.4f}")
print(f"Test  accuracy: {clf.score(X_test, y_test):.4f}")

## Confusion Matrix

In [ ]:
y_pred = clf.predict(X_test)

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"  TP={cm[1,1]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TN={cm[0,0]}")
print()
print(classification_report(y_test, y_pred, target_names=class_names))

fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(class_names); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=14,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.show()

## Feature Importance (Approximation)

We measure how much each feature changes test accuracy when its values are shuffled (permutation importance).

In [ ]:
baseline = clf.score(X_test, y_test)
importances = []
rng = np.random.default_rng(42)

for f in range(X_test.shape[1]):
    X_perm = X_test.copy()
    X_perm[:, f] = rng.permutation(X_perm[:, f])
    importances.append(baseline - clf.score(X_perm, y_test))

importances = np.array(importances)
top_idx = np.argsort(importances)[-10:]

plt.figure(figsize=(9,5))
plt.barh(feature_names[top_idx], importances[top_idx], color='steelblue', edgecolor='white')
plt.xlabel("Accuracy Drop When Feature is Shuffled")
plt.title("Top 10 Feature Importances (Permutation)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

- Decision trees are interpretable and require minimal preprocessing.
- **max_depth** is the primary regularisation knob — too deep = overfitting.
- Entropy + information gain guides each binary split.
- On Breast Cancer, depth=4 achieves >95% test accuracy.
